# **Deteksi Ujaran Kebencian Multi-Label pada Twitter Menggunakan Hierarchical BiLSTM dengan Attention Mechanism**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

### **Deskripsi Proyek**
Notebook ini mengimplementasikan model Deep Learning **Hierarchical Bidirectional Long Short-Term Memory (Hierarchical BiLSTM)** yang dilengkapi dengan **Attention Mechanism** (*Hierarchical Attention Network* / HAN) untuk mendeteksi ujaran kebencian multi-label pada data Twitter berbahasa Indonesia.

### **4 Target Kategori Multi-Label:**
1. `HS_Religion` : Ujaran kebencian berbasis agama / keyakinan.
2. `HS_Physical` : Ujaran kebencian terkait fisik / disabilitas.
3. `HS_Individual` : Ujaran kebencian yang ditargetkan kepada individu / perorangan.
4. `HS_Group` : Ujaran kebencian yang ditargetkan kepada kelompok / golongan tertentu.

### **Fitur Unggulan Notebook Ini:**
- **Data Augmentation:** Menangani ketimpangan kelas (*class imbalance*) pada kelas minoritas (`HS_Physical` dan `HS_Religion`) menggunakan teknik *Easy Data Augmentation (EDA)* pada data training.
- **Hierarchical Structure (Word & Sentence Level):** Menangkap struktur tweet yang berjenjang (kata $\rightarrow$ kalimat $\rightarrow$ tweet).
- **Trainable Attention Mechanism:** Menyediakan *explainability* berupa visualisasi heatmap kata mana yang paling diperhatikan model.
- **Multi-Label Inference:** Sistem mampu mendeteksi lebih dari satu label kebencian sekaligus dalam satu tweet (contoh: Agama + Kelompok).

In [ ]:
# ====================================================================
# 1. SETUP ENVIRONMENT GOOGLE COLAB / LOCAL
# ====================================================================
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    print("💻 Berjalan di Google Colab. Menyiapkan Google Drive...")
    drive.mount('/content/drive')
    
    # Sesuaikan path folder kerja Anda di Google Drive jika diperlukan
    # Contoh: /content/drive/MyDrive/hate_speech/
    # os.chdir('/content/drive/MyDrive/hate_speech/')
except ImportError:
    IN_COLAB = False
    print("💻 Berjalan di Lingkungan Lokal (Jupyter Notebook).")

# Cek Ketersediaan Akselerasi GPU
import tensorflow as tf
print(f"\nVersi TensorFlow : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Terdeteksi: {gpus}")
    print("Akselerasi GPU aktif untuk mempercepat proses training!")
else:
    print("⚠️ GPU tidak terdeteksi. Training akan berjalan pada CPU.")
    if IN_COLAB:
        print("💡 Tips di Colab: Klik Menu 'Runtime' -> 'Change runtime type' -> Pilih 'T4 GPU'.")


In [ ]:
# ====================================================================
# 2. IMPORT LIBRARIES
# ====================================================================
import re
import json
import random
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense, Dropout,
    SpatialDropout1D, TimeDistributed, Layer
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, hamming_loss, accuracy_score, f1_score,
    precision_score, recall_score, multilabel_confusion_matrix, confusion_matrix
)

# Set style dan random seed untuk konsistensi hasil
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print("✅ Seluruh library berhasil diimpor dengan sukses.")


## **Tahap 1: Eksplorasi & Pemuatan Data (`data.csv`)**
Pada tahap ini:
1. Membaca berkas `data.csv`.
2. Menghapus data tweet duplikat agar model tidak bias.
3. Memilih 4 label target: `['HS_Religion', 'HS_Physical', 'HS_Individual', 'HS_Group']`.
4. Melakukan visualisasi distribusi frekuensi tiap label serta korelasi antar label.

In [ ]:
# ====================================================================
# 3. MEMUAT DATASET & MEMILIH 4 LABEL TARGET
# ====================================================================
DATASET_PATH = 'data.csv'
KAMUS_ALAY_PATH = 'new_kamusalay.csv'

# Membaca dataset dengan fallback encoding latin-1
df_raw = pd.read_csv(DATASET_PATH, encoding='latin-1')
print(f"Dimensi data awal: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")

# Drop duplikat teks tweet
df = df_raw.drop_duplicates(subset=['Tweet']).reset_index(drop=True)
print(f"Dimensi setelah menghapus duplikat: {df.shape[0]} baris\n")

# Tentukan 4 target label multi-label
TARGET_LABELS = ['HS_Religion', 'HS_Physical', 'HS_Individual', 'HS_Group']

print("=== Preview 5 Baris Data ===")
display(df[['Tweet'] + TARGET_LABELS].head())

print("\n=== Distribusi Frekuensi 4 Label Target ===")
dist_df = pd.DataFrame({
    'Label': TARGET_LABELS,
    'Jumlah Positif': [df[col].sum() for col in TARGET_LABELS],
    'Persentase (%)': [(df[col].sum() / len(df)) * 100 for col in TARGET_LABELS]
})
display(dist_df)


In [ ]:
# ====================================================================
# 4. VISUALISASI EKSPLORASI DATA (EDA)
# ====================================================================
plt.figure(figsize=(14, 5))

# 1. Bar plot frekuensi setiap label
plt.subplot(1, 2, 1)
ax = sns.barplot(x='Label', y='Jumlah Positif', data=dist_df, palette='Blues_r')
plt.title('Distribusi 4 Kategori Ujaran Kebencian (Sebelum Augmentasi)', fontsize=13, fontweight='bold')
plt.xlabel('Label Target')
plt.ylabel('Jumlah Tweet')
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=10, color='black', xytext=(0, 4),
                textcoords='offset points')

# 2. Heatmap korelasi antar 4 label
plt.subplot(1, 2, 2)
corr = df[TARGET_LABELS].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", cbar=True, vmin=-0.5, vmax=1.0)
plt.title('Korelasi Antar Label (Multi-Label Relationship)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Menampilkan jumlah kombinasi label per tweet
df['total_labels'] = df[TARGET_LABELS].sum(axis=1)
label_counts = df['total_labels'].value_counts().sort_index()
print("Distribusi Jumlah Label Aktif per Tweet:")
for count, freq in label_counts.items():
    pct = (freq / len(df)) * 100
    print(f"- Memiliki {count} label aktif : {freq} tweet ({pct:.2f}%)")


## **Tahap 2: Pembersihan Teks, Normalisasi Alay & Sentence Splitting**

### **Strategi Preprocessing:**
1. **Case Folding:** Mengubah semua teks menjadi huruf kecil (*lowercase*).
2. **Regex Cleaning:** Menghapus tautan URL, mention pengguna (`@user` / `USER`), karakter non-ASCII (emoji), angka, dan tanda baca yang tidak diperlukan.
3. **Normalisasi Kata Alay / Slang (`new_kamusalay.csv`):** Mengganti kata tidak baku / singkatan gaul menjadi kata baku bahasa Indonesia (contoh: *loe* $\rightarrow$ *kamu*, *yg* $\rightarrow$ *yang*, *bego* $\rightarrow$ *bodoh*).
4. **Sentence Splitting:** Karena model ini berbasis **Hierarchical Attention Network (HAN)**, setiap tweet dipisahkan menjadi kalimat-kalimat berdasarkan tanda pemisah seperti titik (`.`), tanda seru (`!`), tanda tanya (`?`), atau enter (`\n`).

In [ ]:
# ====================================================================
# 5. IMPLEMENTASI PREPROCESSING & NORMALISASI KAMUS ALAY
# ====================================================================
# Muat Kamus Alay jika tersedia
alay_dict = {}
if os.path.exists(KAMUS_ALAY_PATH):
    df_alay = pd.read_csv(KAMUS_ALAY_PATH, encoding='latin-1', header=None)
    alay_dict = dict(zip(df_alay[0], df_alay[1]))
    print(f"✅ Berhasil memuat {len(alay_dict)} entri kamus alay dari {KAMUS_ALAY_PATH}")
else:
    print(f"⚠️ Berkas {KAMUS_ALAY_PATH} tidak ditemukan. Preprocessing tetap berjalan tanpa mapping alay.")

def clean_and_split_sentences(text):
    """
    Membersihkan teks tweet dan memecahnya menjadi daftar kalimat yang telah dinormalisasi.
    """
    text = str(text).lower()
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+|url\b', ' ', text)
    # Hapus Mention / User
    text = re.sub(r'@\w+|user\b', ' ', text)
    # Hapus tanda hashtag (#) tetapi pertahankan katanya (#jokowi -> jokowi)
    text = re.sub(r'#(\w+)', r' \1 ', text)
    # Hapus karakter non-ASCII (emoji, simbol aneh)
    text = text.encode('ascii', 'ignore').decode('utf-8')
    # Reduksi huruf berulang lebih dari 2 kali (contoh: 'begooolll' -> 'begool')
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Pisahkan tweet menjadi kalimat berdasarkan tanda baca atau enter
    raw_sentences = re.split(r'[\.\!\?\n;]+|\\\\n|\\n', text)
    clean_sentences = []
    
    for sent in raw_sentences:
        # Ambil hanya karakter alfabet
        words = re.findall(r'[a-zA-Z]+', sent)
        if words:
            # Normalisasi kata alay menggunakan kamus
            norm_words = [alay_dict.get(w, w) for w in words]
            clean_sentences.append(' '.join(norm_words))
            
    # Jika tweet kosong setelah dibersihkan, isi dengan string kosong
    if not clean_sentences:
        clean_sentences = ['']
    return clean_sentences

print("Sedang melakukan preprocessing pada seluruh dataset...")
df['sentences'] = df['Tweet'].apply(clean_and_split_sentences)
df['sentence_count'] = df['sentences'].apply(len)
print(f"✅ Selesai! Statistik jumlah kalimat per tweet:")
print(df['sentence_count'].describe(percentiles=[0.5, 0.75, 0.9, 0.95]))

print("\n=== Contoh Hasil Preprocessing ===")
for i in range(3):
    print(f"Tweet Asli #{i+1} :", df['Tweet'].iloc[i])
    print(f"Kalimat Terpisah :", df['sentences'].iloc[i])
    print("-" * 80)


## **Tahap 3: Pembagian Dataset (Train 70%, Validation 15%, Test 15%)**

> [!IMPORTANT]
> Pembagian dataset dilakukan **SEBELUM** augmentasi data. Ini memastikan bahwa data Validation dan Testing murni berasal dari data asli (mencegah *Data Leakage*).

In [ ]:
# ====================================================================
# 6. PEMBAGIAN DATASET (TRAIN 70%, VALIDATION 15%, TEST 15%)
# ====================================================================
# Split pertama: Pisahkan Test set (15%)
df_temp, df_test = train_test_split(
    df, test_size=0.15, random_state=RANDOM_SEED
)

# Split kedua: Pisahkan Train (70% total) dan Validation (15% total)
df_train, df_val = train_test_split(
    df_temp, test_size=(0.15 / 0.85), random_state=RANDOM_SEED
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print(f"Jumlah Data Training Awal : {len(df_train)} sampel ({len(df_train)/len(df)*100:.1f}%)")
print(f"Jumlah Data Validation    : {len(df_val)} sampel ({len(df_val)/len(df)*100:.1f}%)")
print(f"Jumlah Data Testing       : {len(df_test)} sampel ({len(df_test)/len(df)*100:.1f}%)\n")

split_summary = pd.DataFrame({
    'Label': TARGET_LABELS,
    'Train (%)': (df_train[TARGET_LABELS].sum().values / len(df_train)) * 100,
    'Val (%)': (df_val[TARGET_LABELS].sum().values / len(df_val)) * 100,
    'Test (%)': (df_test[TARGET_LABELS].sum().values / len(df_test)) * 100
})
print("Proporsi Label Positif di Setiap Split (Sebelum Augmentasi):")
display(split_summary.round(2))


## **Tahap 4: Augmentasi Data Teks Khusus Kelas Minoritas (Easy Data Augmentation - EDA)**

### **Mengapa Augmentasi Diperlukan?**
Pada data training, label `HS_Physical` hanya memiliki ~222 sampel (2.4%) dan `HS_Religion` hanya memiliki ~549 sampel (6.0%). Tanpa penanganan, model BiLSTM cenderung malas memprediksi label minoritas ini.

### **Teknik Augmentasi Teks (EDA - Wei & Zou, 2019):**
1. **Random Swap (RS):** Menukar posisi dua kata acak dalam kalimat untuk melatih model memahami variasi struktur kalimat bebas di Twitter.
2. **Random Deletion (RD):** Menghapus kata non-esensial secara acak dengan probabilitas rendah agar model tangguh terhadap singkatan/kata yang hilang.

### **Aturan Augmentasi:**
- Tweet dengan label **`HS_Physical`** digandakan **3x lipat** (+2 sampel sintetis baru).
- Tweet dengan label **`HS_Religion`** digandakan **2x lipat** (+1 sampel sintetis baru).
- Hanya diaplikasikan pada **Data Training**, data Validation dan Testing **TIDAK diaugmentasi**.

In [ ]:
# ====================================================================
# 7. IMPLEMENTASI DATA AUGMENTATION (EDA) PADA DATA TRAINING
# ====================================================================
def augment_sentences_eda(sentences, p_swap=0.2, p_delete=0.1):
    """
    Menerapkan Random Swap dan Random Deletion pada kalimat.
    """
    aug_sentences = []
    for sent in sentences:
        words = sent.split()
        if len(words) <= 3:
            aug_sentences.append(sent)
            continue
            
        words_copy = words.copy()
        
        # 1. Random Swap: Tukar 2 kata acak
        if random.random() < 0.7:
            idx1, idx2 = random.sample(range(len(words_copy)), 2)
            words_copy[idx1], words_copy[idx2] = words_copy[idx2], words_copy[idx1]
            
        # 2. Random Deletion: Hapus 1 kata dengan probabilitas p_delete
        remaining_words = [w for w in words_copy if random.random() > p_delete]
        if len(remaining_words) < 2:
            remaining_words = words_copy
            
        aug_sentences.append(' '.join(remaining_words))
        
    return aug_sentences

# Terapkan augmentasi hanya pada data training kelas minoritas
augmented_rows = []
for idx, row in df_train.iterrows():
    is_physical = (row['HS_Physical'] == 1)
    is_religion = (row['HS_Religion'] == 1)
    
    if is_physical:
        # Gandakan 3x lipat (+2 sampel sintetis)
        for _ in range(2):
            new_row = row.copy()
            new_row['sentences'] = augment_sentences_eda(row['sentences'])
            augmented_rows.append(new_row)
    elif is_religion:
        # Gandakan 2x lipat (+1 sampel sintetis)
        new_row = row.copy()
        new_row['sentences'] = augment_sentences_eda(row['sentences'])
        augmented_rows.append(new_row)

df_augmented_samples = pd.DataFrame(augmented_rows)
df_train_aug = pd.concat([df_train, df_augmented_samples], ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"✅ Berhasil menambahkan {len(df_augmented_samples)} tweet sintetis hasil augmentasi!")
print(f"Total Data Training Setelah Augmentasi: {len(df_train_aug)} baris (sebelumnya {len(df_train)} baris)\n")

# Perbandingan Distribusi Sebelum vs Sesudah Augmentasi
aug_comparison = pd.DataFrame({
    'Label': TARGET_LABELS,
    'Sebelum Augmentasi': df_train[TARGET_LABELS].sum().values,
    'Setelah Augmentasi': df_train_aug[TARGET_LABELS].sum().values,
    'Kenaikan (Baris)': (df_train_aug[TARGET_LABELS].sum().values - df_train[TARGET_LABELS].sum().values),
    'Proporsi Akhir (%)': (df_train_aug[TARGET_LABELS].sum().values / len(df_train_aug)) * 100
})
display(aug_comparison)

# Visualisasi Perbandingan Bar Chart
plt.figure(figsize=(10, 5))
plot_df = aug_comparison.melt(id_vars='Label', value_vars=['Sebelum Augmentasi', 'Setelah Augmentasi'], var_name='Kondisi', value_name='Jumlah Tweet')
sns.barplot(x='Label', y='Jumlah Tweet', hue='Kondisi', data=plot_df, palette='Blues')
plt.title('Perbandingan Jumlah Data Training Sebelum vs Sesudah Augmentasi', fontsize=13, fontweight='bold')
plt.show()


## **Tahap 5: Pembentukan Struktur Hierarkis 3D (Document $\rightarrow$ Sentence $\rightarrow$ Word)**

Arsitektur Hierarchical BiLSTM membutuhkan input berdimensi 3:
$$\text{Shape} = (N, \text{MAX\_SENTENCES}, \text{MAX\_WORDS})$$
- **$N$**: Jumlah tweet / sampel data.
- **$\text{MAX\_SENTENCES} = 5$**: Maksimum 5 kalimat per tweet.
- **$\text{MAX\_WORDS} = 35$**: Maksimum 35 kata per kalimat.

In [ ]:
# ====================================================================
# 8. TOKENISASI & PEMBENTUKAN TENSOR 3D (N, MAX_SENTENCES, MAX_WORDS)
# ====================================================================
MAX_SENTENCES = 5
MAX_WORDS = 35

# Fitting Tokenizer hanya pada data Training (termasuk hasil augmentasi)
all_train_sentences = [sent for sents in df_train_aug['sentences'] for sent in sents if sent.strip()]

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(all_train_sentences)
vocab_size = len(tokenizer.word_index) + 1
print(f"Total Kosakata Unik (Vocab Size): {vocab_size} kata")

# Simpan tokenizer untuk kebutuhan inferensi
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("✅ Tokenizer berhasil disimpan ke 'tokenizer.pickle'.")

def encode_hierarchical_data(sentences_list, tokenizer, max_sents, max_words):
    """
    Mengonversi daftar kalimat setiap tweet menjadi array 3D: (num_samples, max_sents, max_words)
    """
    num_samples = len(sentences_list)
    data_tensor = np.zeros((num_samples, max_sents, max_words), dtype='int32')
    
    for i, sents in enumerate(sentences_list):
        sents_to_process = sents[:max_sents]
        seqs = tokenizer.texts_to_sequences(sents_to_process)
        padded_seqs = pad_sequences(seqs, maxlen=max_words, padding='post', truncating='post')
        
        for j, sent_seq in enumerate(padded_seqs):
            data_tensor[i, j, :] = sent_seq
            
    return data_tensor

# Konversi Train, Validation, dan Test ke format 3D Tensor
X_train = encode_hierarchical_data(df_train_aug['sentences'], tokenizer, MAX_SENTENCES, MAX_WORDS)
y_train = df_train_aug[TARGET_LABELS].values.astype(np.float32)

X_val = encode_hierarchical_data(df_val['sentences'], tokenizer, MAX_SENTENCES, MAX_WORDS)
y_val = df_val[TARGET_LABELS].values.astype(np.float32)

X_test = encode_hierarchical_data(df_test['sentences'], tokenizer, MAX_SENTENCES, MAX_WORDS)
y_test = df_test[TARGET_LABELS].values.astype(np.float32)

print(f"\nShape X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"Shape X_val   : {X_val.shape}   | y_val   : {y_val.shape}")
print(f"Shape X_test  : {X_test.shape}  | y_test  : {y_test.shape}")


## **Tahap 6: Konfigurasi Word Embedding**

### **Mengapa Trainable Embedding Direkomendasikan untuk Twitter?**
1. **Karakteristik Teks Media Sosial:** Tweet Indonesia mengandung banyak variasi bahasa santai (*slang*, singkatan, gaya bahasa netizen). Meskipun sudah dinormalisasi dengan kamus alay, representasi vektor yang dilatih langsung (*trainable embedding 128-dimensi*) dapat menangkap konotasi ujaran kebencian secara spesifik pada domain Twitter.
2. **Efisiensi Memori & Bebas Download:** Model Pre-trained seperti FastText berukuran > 1.2 GB yang membutuhkan alokasi RAM dan waktu download cukup lama di Google Colab. Trainable Embedding 128 dimensi berjalan sangat cepat, ringan, dan memiliki akurasi tinggi.

## **Tahap 7: Arsitektur Hierarchical BiLSTM dengan Attention Mechanism (HAN)**

### **Attention Mechanism Formulasi (Yang et al., 2016):**
Untuk setiap representasi $h_{it}$ dari BiLSTM:
$$u_{it} = \tanh(W h_{it} + b)$$
$$\alpha_{it} = \frac{\exp(u_{it}^T u_w)}{\sum_t \exp(u_{it}^T u_w)}$$
$$s_i = \sum_t \alpha_{it} h_{it}$$
Di mana $u_w$ adalah **Context Vector** yang dipelajari selama proses training.

In [ ]:
# ====================================================================
# 9. CUSTOM LAYER: HIERARCHICAL ATTENTION MECHANISM
# ====================================================================
class HierarchicalAttention(Layer):
    """
    Attention Mechanism dengan Trainable Context Vector (Yang et al., 2016)
    Dapat digunakan baik pada Word-level maupun Sentence-level.
    """
    def __init__(self, return_attention=False, **kwargs):
        super(HierarchicalAttention, self).__init__(**kwargs)
        self.return_attention = return_attention

    def build(self, input_shape):
        feature_dim = input_shape[-1]
        self.W = self.add_weight(
            name="att_weight",
            shape=(feature_dim, feature_dim),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(feature_dim,),
            initializer="zeros",
            trainable=True
        )
        self.u = self.add_weight(
            name="context_vector",
            shape=(feature_dim, 1),
            initializer="glorot_uniform",
            trainable=True
        )
        super(HierarchicalAttention, self).build(input_shape)

    def call(self, inputs, mask=None):
        uit = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)
        ait = tf.tensordot(uit, self.u, axes=1)
        ait = tf.squeeze(ait, -1)
        
        if mask is not None:
            mask = tf.cast(mask, tf.float32)
            ait -= (1.0 - mask) * 1e9
            
        ait = tf.nn.softmax(ait, axis=-1)
        ait_expanded = tf.expand_dims(ait, -1)
        output = tf.reduce_sum(inputs * ait_expanded, axis=1)
        
        if self.return_attention:
            return output, ait
        return output

    def get_config(self):
        config = super(HierarchicalAttention, self).get_config()
        config.update({"return_attention": self.return_attention})
        return config

print("✅ Custom HierarchicalAttention Layer siap digunakan.")


In [ ]:
# ====================================================================
# 10. MEMBANGUN MODEL HIERARCHICAL BiLSTM DENGAN ATTENTION
# ====================================================================
EMBEDDING_DIM = 128
WORD_LSTM_UNITS = 64
SENT_LSTM_UNITS = 64
NUM_CLASSES = len(TARGET_LABELS)  # 4 Kategori

# --------------------------------------------------
# SUB-MODEL 1: WORD-LEVEL ENCODER (Representasi Kalimat)
# --------------------------------------------------
word_input = Input(shape=(MAX_WORDS,), dtype='int32', name='word_input')
word_embed = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    name='word_embedding'
)(word_input)
word_embed_drop = SpatialDropout1D(0.2, name='word_spatial_dropout')(word_embed)
word_bilstm = Bidirectional(
    LSTM(WORD_LSTM_UNITS, return_sequences=True),
    name='word_bilstm'
)(word_embed_drop)
sentence_vector = HierarchicalAttention(name='word_attention')(word_bilstm)

sentence_encoder = Model(word_input, sentence_vector, name='Sentence_Encoder')
print("--- Sentence Encoder (Word Level) ---")
sentence_encoder.summary()

# --------------------------------------------------
# MAIN MODEL 2: SENTENCE-LEVEL ENCODER (Dokumen Tweet)
# --------------------------------------------------
doc_input = Input(shape=(MAX_SENTENCES, MAX_WORDS), dtype='int32', name='document_input')
doc_encoded = TimeDistributed(sentence_encoder, name='time_distributed_sentences')(doc_input)
sent_bilstm = Bidirectional(
    LSTM(SENT_LSTM_UNITS, return_sequences=True),
    name='sentence_bilstm'
)(doc_encoded)
doc_vector = HierarchicalAttention(name='sentence_attention')(sent_bilstm)

# --------------------------------------------------
# CLASSIFICATION HEAD: MULTI-LABEL CLASSIFIER
# --------------------------------------------------
dense_1 = Dense(64, activation='relu', name='dense_classifier')(doc_vector)
drop_1 = Dropout(0.3, name='classifier_dropout')(dense_1)
output_layer = Dense(NUM_CLASSES, activation='sigmoid', name='multilabel_output')(drop_1)

model = Model(inputs=doc_input, outputs=output_layer, name='Hierarchical_BiLSTM_Attention_MultiLabel')

# Kompilasi Model dengan Binary Crossentropy Loss & Adam Optimizer
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(name='auc')
    ]
)

print("\n--- Ringkasan Arsitektur Model Utama ---")
model.summary()


In [ ]:
# ====================================================================
# 11. TRAINING MODEL DENGAN DATASET TER-AUGMENTASI
# ====================================================================
EPOCHS = 20
BATCH_SIZE = 64

callbacks = [
    ModelCheckpoint(
        'best_hierarchical_bilstm_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

print("🚀 Memulai pelatihan model dengan data training ter-augmentasi...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print("✅ Pelatihan selesai!")


In [ ]:
# ====================================================================
# 12. VISUALISASI KURVA TRAINING VS VALIDATION
# ====================================================================
plt.figure(figsize=(18, 5))

# 1. Plot Loss
plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='Train Loss', marker='o', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', marker='s', linewidth=2)
plt.title('Kurva Loss (Binary Crossentropy)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# 2. Plot Binary Accuracy
plt.subplot(1, 3, 2)
plt.plot(history.history['binary_accuracy'], label='Train Accuracy', marker='o', linewidth=2)
plt.plot(history.history['val_binary_accuracy'], label='Val Accuracy', marker='s', linewidth=2)
plt.title('Kurva Binary Accuracy', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# 3. Plot AUC
plt.subplot(1, 3, 3)
plt.plot(history.history['auc'], label='Train AUC', marker='o', linewidth=2)
plt.plot(history.history['val_auc'], label='Val AUC', marker='s', linewidth=2)
plt.title('Kurva AUC (Area Under ROC)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()

plt.tight_layout()
plt.show()


## **Tahap 10: Evaluasi Multi-Label pada Data Test**

Metrik evaluasi multi-label:
1. **Hamming Loss:** Proporsi label yang salah diprediksi secara keseluruhan (semakin kecil semakin baik).
2. **Subset Accuracy (Exact Match):** Persentase di mana seluruh 4 label berhasil diprediksi tepat secara bersamaan.
3. **Macro F1-Score:** Rata-rata F1 per kelas tanpa pembobotan frekuensi (sangat sensitif terhadap kelas minoritas).
4. **Threshold Optimization:** Menyesuaikan threshold per kelas berdasarkan skor F1 terbaik pada data validasi.

In [ ]:
# ====================================================================
# 13. EVALUASI METRIK MULTI-LABEL PADA DATA TESTING
# ====================================================================
y_pred_prob = model.predict(X_test)

# Binarisasi dengan threshold default 0.50
THRESHOLD = 0.5
y_pred = (y_pred_prob >= THRESHOLD).astype(int)

hl = hamming_loss(y_test, y_pred)
exact_acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
micro_f1 = f1_score(y_test, y_pred, average='micro', zero_division=0)
macro_prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
macro_rec = recall_score(y_test, y_pred, average='macro', zero_division=0)

print("=" * 60)
print("    RINGKASAN HASIL EVALUASI MULTI-LABEL (THRESHOLD 0.5)    ")
print("=" * 60)
print(f"Hamming Loss (Lebih kecil lebih baik) : {hl:.4f}")
print(f"Exact Match Ratio (Subset Accuracy)   : {exact_acc * 100:.2f}%")
print(f"Macro-Precision                        : {macro_prec:.4f}")
print(f"Macro-Recall                           : {macro_rec:.4f}")
print(f"Macro F1-Score                         : {macro_f1:.4f}")
print(f"Micro F1-Score                         : {micro_f1:.4f}")
print("=" * 60)

print("\n=== Detail Classification Report (Threshold 0.50) ===")
print(classification_report(y_test, y_pred, target_names=TARGET_LABELS, zero_division=0, digits=4))

# --------------------------------------------------------------------
# PENCARIAN THRESHOLD OPTIMAL PER KELAS PADA DATA VALIDASI
# --------------------------------------------------------------------
y_val_prob = model.predict(X_val, verbose=0)
optimal_thresholds = {}
print("\n=== Pencarian Threshold Optimal per Kategori (Tuning F1-Score) ===")
for i, label in enumerate(TARGET_LABELS):
    best_th = 0.5
    best_f1 = 0.0
    for th in np.arange(0.10, 0.85, 0.02):
        f1 = f1_score(y_val[:, i], (y_val_prob[:, i] >= th).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_th = round(float(th), 2)
    optimal_thresholds[label] = best_th
    print(f"- {label:15}: Best Threshold = {best_th:.2f} (F1 Val = {best_f1:.4f})")

# Evaluasi ulang data Test dengan Threshold Optimal
y_pred_opt = np.zeros_like(y_test, dtype=int)
for i, label in enumerate(TARGET_LABELS):
    y_pred_opt[:, i] = (y_pred_prob[:, i] >= optimal_thresholds[label]).astype(int)

macro_f1_opt = f1_score(y_test, y_pred_opt, average='macro', zero_division=0)
print(f"\nPerbandingan Macro F1-Score:")
print(f"- Default (Threshold 0.50) : {macro_f1:.4f}")
print(f"- Optimal  (Tuned Thresh)  : {macro_f1_opt:.4f}")


In [ ]:
# ====================================================================
# 14. VISUALISASI CONFUSION MATRIX UNTUK SETIAP LABEL
# ====================================================================
mcm = multilabel_confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.ravel()

for idx, label_name in enumerate(TARGET_LABELS):
    cm = mcm[idx]
    sns.heatmap(
        cm, annot=True, fmt="d", cmap='Blues', cbar=False, ax=axes[idx],
        annot_kws={"size": 13, "weight": "bold"},
        xticklabels=['Negatif (0)', 'Positif (1)'],
        yticklabels=['Negatif (0)', 'Positif (1)']
    )
    axes[idx].set_title(f'Confusion Matrix: {label_name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Prediksi Model')
    axes[idx].set_ylabel('Label Sebenarnya')

plt.tight_layout()
plt.show()


## **Tahap 11: Visualisasi Bobot Attention (Interpretability / Explainability)**

Ekstraksi bobot perhatian kata (*Word Attention*) untuk menjelaskan kata mana yang paling diperhatikan model saat mendeteksi ujaran kebencian.

In [ ]:
# ====================================================================
# 15. EKSTRAKSI DAN VISUALISASI BOBOT ATTENTION KATA
# ====================================================================
try:
    sent_enc = sentence_encoder
except NameError:
    sent_enc = model.get_layer('time_distributed_sentences').layer

word_bilstm_layer = sent_enc.get_layer('word_bilstm')
word_att_layer = sent_enc.get_layer('word_attention')

word_lstm_submodel = Model(
    inputs=sent_enc.input,
    outputs=word_bilstm_layer.output,
    name='Word_BiLSTM_Extractor'
)

def visualize_word_attention(sentence_text):
    """
    Visualisasi bobot perhatian kata (Word Attention) pada satu kalimat tertentu.
    """
    words = sentence_text.split()[:MAX_WORDS]
    if not words:
        return
    seq = tokenizer.texts_to_sequences([' '.join(words)])
    padded = pad_sequences(seq, maxlen=MAX_WORDS, padding='post', truncating='post')
    
    # 1. Hidden states dari Word BiLSTM: shape (1, MAX_WORDS, 2*units)
    h = word_lstm_submodel.predict(padded, verbose=0)
    
    # 2. Ambil bobot terlatih dari layer Attention (W, b, u)
    W, b, u = word_att_layer.get_weights()
    
    # 3. Hitung scoring attention: uit = tanh(h * W + b), ait = uit * u
    uit = np.tanh(np.dot(h, W) + b)
    ait = np.dot(uit, u).squeeze(-1)  # shape: (1, MAX_WORDS)
    
    # 4. Softmax pada kata aktual
    valid_ait = ait[0][:len(words)]
    exp_ait = np.exp(valid_ait - np.max(valid_ait))
    valid_alphas = exp_ait / np.sum(exp_ait)
    
    # 5. Visualisasi Heatmap
    plt.figure(figsize=(max(8, len(words) * 0.9), 2))
    sns.heatmap(
        valid_alphas.reshape(1, -1), annot=True, fmt=".2f", cmap='YlOrRd',
        xticklabels=words, yticklabels=['Perhatian'], cbar=True
    )
    plt.title("Word-Level Attention Heatmap", fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.show()

# Uji coba visualisasi bobot perhatian pada contoh kalimat uji
sample_test_sentence = "dasar orang kafir perusak agama jangan kamu biarkan"
print(f"Contoh Kalimat: '{sample_test_sentence}'")
visualize_word_attention(sample_test_sentence)


## **Tahap 12: Pengujian Interaktif Tweet Baru (Multi-Label Inference)**

Fungsi `predict_tweet()` siap pakai untuk mendeteksi berbagai kombinasi multi-label secara simultan dalam satu tweet.

In [ ]:
# ====================================================================
# 16. FUNGSI PREDIKSI MULTI-LABEL PADA TWEET BARU
# ====================================================================
DEFAULT_THRESHOLDS = {
    'HS_Religion': 0.35,
    'HS_Physical': 0.25,
    'HS_Individual': 0.45,
    'HS_Group': 0.45
}

def predict_tweet(raw_tweet, thresholds=None, verbose=True):
    """
    Menerima tweet mentah, melakukan pembersihan teks, encoding 3D,
    dan mendeteksi multi-label secara simultan.
    """
    if thresholds is None:
        thresholds = optimal_thresholds if 'optimal_thresholds' in globals() else DEFAULT_THRESHOLDS
        
    # 1. Preprocessing & Sentence Splitting
    sents = clean_and_split_sentences(raw_tweet)
    
    # 2. Encoding ke struktur hierarki 3D (1, MAX_SENTENCES, MAX_WORDS)
    tensor_input = encode_hierarchical_data([sents], tokenizer, MAX_SENTENCES, MAX_WORDS)
    
    # 3. Model Inference
    probs = model.predict(tensor_input, verbose=0)[0]
    
    results = []
    detected_labels = []
    for i, label in enumerate(TARGET_LABELS):
        th = thresholds[label] if isinstance(thresholds, dict) else thresholds
        is_detected = (probs[i] >= th)
        if is_detected:
            detected_labels.append(label)
            
        results.append({
            'Kategori': label,
            'Probabilitas': f"{probs[i] * 100:.2f}%",
            'Threshold': f"{th * 100:.0f}%",
            'Status': '✅ YA (Terdeteksi)' if is_detected else '❌ TIDAK (Negatif)'
        })
        
    df_res = pd.DataFrame(results)
    
    if verbose:
        print("=" * 75)
        print(f"📝 Tweet Input        : \"{raw_tweet}\"")
        print(f"🔍 Kalimat Terurai    : {sents}")
        if detected_labels:
            print(f"🚨 Multi-Label Terdeteksi ({len(detected_labels)} Kategori): {', '.join(detected_labels)}")
        else:
            print("🟢 Multi-Label Terdeteksi (0 Kategori): Aman / Tidak Ada Ujaran Kebencian")
        display(df_res[['Kategori', 'Probabilitas', 'Threshold', 'Status']])
        
    return df_res

# --------------------------------------------------------------------
# PENGUJIAN CONTOH KASUS MULTI-LABEL TWEET (UNSEEN DATA)
# --------------------------------------------------------------------
test_cases_unseen = [
    # Kasus 1: Tweet netral / opini publik biasa (Harus 0 Label)
    "Menurut saya kebijakan transportasi umum saat ini sudah jauh lebih baik dibandingkan tahun lalu. Mari kita kawal bersama pembangunannya.",
    
    # Kasus 2: Multi-label Agama + Kelompok (HS_Religion DAN HS_Group)
    "Usir semua penganut sekte sesat itu dari kampung ini! Ajaran agama kalian najis dan kelompok kalian cuma bikin onar!",
    
    # Kasus 3: Multi-label Fisik + Individu (HS_Physical DAN HS_Individual)
    "Heh cebol cacat, jalan aja masih pincang gak usah sok jagoan lu di Twitter!",
    
    # Kasus 4: Hinaan Individu (HS_Individual saja)
    "Dasar koruptor bajingan tidak tahu malu, semoga kamu cepat membusuk di penjara!",
    
    # Kasus 5: Multi-label Fisik + Kelompok (HS_Physical DAN HS_Group)
    "Kumpulan orang-orang idiot dan sakit jiwa emang mereka itu, satu gerombolan otaknya miring semua!",
    
    # Kasus 6: Multi-label Agama + Individu (HS_Religion DAN HS_Individual)
    "Lu murtad kafir munafik, jangan bawa-bawa ayat suci untuk membenarkan kelakuan bejat lu!",
    
    # Kasus 7: Tweet komplain/kesal biasa tanpa ujaran kebencian (Harus 0 Label)
    "Pelayanan aplikasi ini lambat banget parah, bikin emosi tiap kali mau transaksi error terus!"
]

print("=== PENGUJIAN CONTOH TWEET BARU (UNSEEN MULTI-LABEL) ===")
for tweet in test_cases_unseen:
    predict_tweet(tweet)


In [ ]:
# ====================================================================
# 17. PENYIMPANAN MODEL, TOKENIZER, & KONFIGURASI PROYEK
# ====================================================================
MODEL_SAVE_PATH = 'hierarchical_bilstm_attention_model.keras'
CONFIG_SAVE_PATH = 'config_model.json'

# 1. Simpan Model Akhir
model.save(MODEL_SAVE_PATH)
print(f"✅ Model berhasil disimpan ke '{MODEL_SAVE_PATH}'.")

# 2. Simpan file konfigurasi parameter
config_data = {
    'max_sentences': MAX_SENTENCES,
    'max_words': MAX_WORDS,
    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'word_lstm_units': WORD_LSTM_UNITS,
    'sent_lstm_units': SENT_LSTM_UNITS,
    'target_labels': TARGET_LABELS,
    'random_seed': RANDOM_SEED
}

with open(CONFIG_SAVE_PATH, 'w') as f:
    json.dump(config_data, f, indent=4)
print(f"✅ Konfigurasi berhasil disimpan ke '{CONFIG_SAVE_PATH}'.")

# 3. Opsi download langsung di Google Colab
if IN_COLAB:
    print("\n📥 Anda dapat mendownload file hasil pelatihan ke komputer lokal:")
    print("Contoh perintah download: files.download('best_hierarchical_bilstm_model.keras')")
